# Sweep all inference tasks across many models (Colab driver)

Loads each model **once**, runs every demographic_bias and sycophancy inference
task against it, then unloads and moves to the next. After each model finishes
its task list, runs `build_csv.py` for both experiments to produce the combined
CSVs the R analysis pipeline (`analysis/analysis_*.Rmd`) consumes.

Outputs land in Google Drive. Re-running the driver cell skips (model, config, task)
triples that already have output, so it's safe to resume after a Colab session
timeout.

This is the **thinking-OFF** driver (the canonical sweep). For thinking-on, see
`../run_all_models_colab_thinking.ipynb` in the private repo.

Runtime: a Colab Pro A100 80GB will take roughly an hour for the small Qwen3
sizes and several hours for 14B+. The 32B (8-bit quantized in `model_config.yaml`)
and 30B-A3B (MoE) models are the slowest — plan for an overnight run if
sweeping the full set.

## 1. Clone the repo and install dependencies

In [ ]:
%cd /content
![ -d public_repo ] || git clone https://github.com/self-model/SelfBlindingLLMs.git public_repo

In [ ]:
!pip install -q -r public_repo/requirements.txt
# Needed for any 8-bit quantized entries in model_config.yaml (Qwen3-32B, Qwen2.5-72B)
!pip install -q bitsandbytes
# Fast-path kernels for hybrid/linear-attention models (Qwen3.8 family).
# Without these, transformers warns "The fast path is not available" and falls
# back to a much slower pure-torch implementation. causal-conv1d compiles a
# CUDA extension whose setup.py imports torch, so it needs --no-build-isolation
# (pip's isolated build env has no torch and the wheel build fails otherwise).
!pip install -q flash-linear-attention
!pip install -q causal-conv1d --no-build-isolation

## 2. Mount Google Drive (for persistent outputs)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configure: which models, where to save

Edit `MODELS` to control the sweep. Comment out anything you don't want to run.

All listed models must have an entry in `public_repo/src/model_config.yaml`.
The 235B is intentionally absent — it doesn't fit on a single A100.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, '/content/public_repo')
from src.thinking import ThinkingConfig

# SWEEP is a list of (model_name, ThinkingConfig) tuples.
# Each tuple becomes its own output dir + CSV — thinking-off and thinking-on
# entries are siblings, not overrides of each other.
#
# This notebook defaults to thinking-OFF (the canonical sweep). For the
# thinking-on counterpart, see ../run_all_models_colab_thinking.ipynb in the
# private repo.
SWEEP = [
    ('Qwen/Qwen3-0.6B',  ThinkingConfig(mode='off')),
    ('Qwen/Qwen3-1.7B',  ThinkingConfig(mode='off')),
    ('Qwen/Qwen3-4B',    ThinkingConfig(mode='off')),
    ('Qwen/Qwen3-8B',    ThinkingConfig(mode='off')),
    ('Qwen/Qwen3-14B',   ThinkingConfig(mode='off')),
    ('Qwen/Qwen3-32B',   ThinkingConfig(mode='off')),
    # ('Qwen/Qwen3-30B-A3B', ThinkingConfig(mode='off')),  # MoE — disabled; prohibitively slow at batch=1.
    #                                                       # Re-enable once Tier-3 batched inference lands
    #                                                       # (see ~/.claude/plans/batching-optimization-tiers.md).
]

ROOT = Path('/content/drive/MyDrive/Oxford/self_blinding/outputs')
ROOT.mkdir(parents=True, exist_ok=True)
print(f'Outputs will land under: {ROOT}')
print(f'SWEEP has {len(SWEEP)} (model, config) entries')

## 4. The driver loop

For each model: load weights once, run all 7 inference tasks (saving raw
JSONL to Drive after each), then run `build_csv.py` for both experiments to
produce the combined CSVs, then unload.

Drive layout:
```
{ROOT}/
├── demographic_bias/
│   ├── {model_nickname}/   # raw inference JSONL per task
│   └── processed/          # demographic_bias_processed_{nick}.csv per model
└── sycophancy/
    ├── {model_nickname}/
    └── processed/
```

If a (model, task) output already exists in Drive, it is skipped — so
re-running this cell after a session timeout resumes from where you left off.

In [ ]:
import sys
import subprocess
import traceback
from types import SimpleNamespace

sys.path.insert(0, '/content/public_repo')

import torch
from src.inference import load_model_and_tokenizer
from src.utils import clear_gpu_memory
from src.thinking import ThinkingConfig, get_thinking_family

from demographic_bias.inference import (
    yn_logprobs_hf,
    tool_use_probs_hf as bias_tool_use,
    tool_result_yn_logprobs_hf as bias_tool_result,
)
from sycophancy.inference import (
    first_person_hf,
    third_person_hf,
    tool_use_probs_hf as syc_tool_use,
    tool_result_yn_logprobs_hf as syc_tool_result,
)

# (experiment, output-filename-tag-for-skip-check, module)
TASKS = [
    ('demographic_bias', 'bias_yn',                  yn_logprobs_hf),
    ('demographic_bias', 'bias_tool_use',            bias_tool_use),
    ('demographic_bias', 'bias_tool_result_yn',      bias_tool_result),
    ('sycophancy',       'sycophancy_first_person',  first_person_hf),
    ('sycophancy',       'sycophancy_third_person',  third_person_hf),
    ('sycophancy',       'sycophancy_tool_use',      syc_tool_use),
    ('sycophancy',       'sycophancy_tool_result',   syc_tool_result),
]

BUILD_CSV_SCRIPTS = {
    'demographic_bias': '/content/public_repo/demographic_bias/build_csv.py',
    'sycophancy':       '/content/public_repo/sycophancy/build_csv.py',
}

# Skip-if-exists size floor.
MIN_VALID_OUTPUT_BYTES = 1024


def compute_nickname(model_name: str, cfg: ThinkingConfig) -> str:
    """Full nickname for the (model, config) tuple — used as the Drive subdir
    name and the build_csv --model arg (which sets the output CSV filename).
    The inference scripts continue to use the SHORT nickname (just the model
    basename) inside the JSONL filename, since args.model must remain the real
    HF name."""
    base = model_name.split('/')[-1]
    if cfg.mode == 'off':
        return base
    parts = [base, '_thinkon']
    if cfg.budget != -1:
        parts.append(f'_b{cfg.budget}')
    if cfg.temperature > 0:
        parts.append(f'_T{int(cfg.temperature)}')
    return ''.join(parts)


def make_args(model_name, output_dir, cfg: ThinkingConfig):
    """Superset of args used by all 7 inference scripts (extras are ignored)."""
    return SimpleNamespace(
        model=model_name,
        output_dir=str(output_dir),
        data_path=None,
        tool_prompts_path=None,
        inspect=False,
        inspect_n=3,
        seed=42,
        n_scenarios=None,
        n_letter_pairs=1,
        device=None,
        prefix_cache=False,
        thinking=cfg.mode,
        thinking_budget=cfg.budget,
        thinking_temperature=cfg.temperature,
        thinking_n_samples=cfg.n_samples,
    )


for model_name, cfg in SWEEP:
    short_nick = model_name.split('/')[-1]
    full_nick = compute_nickname(model_name, cfg)
    print(f"\n{'='*70}\nMODEL: {model_name}  |  config: {cfg}  |  nickname: {full_nick}\n{'='*70}")

    # Hard-error guard: skip if thinking-on requested for a model with no thinking_family.
    if cfg.mode == 'on' and get_thinking_family(model_name) is None:
        print(f'  SKIP: thinking-on requested but {model_name} has no thinking_family set in model_config.yaml')
        continue

    print(f'Loading {model_name}...')
    model, tokenizer = load_model_and_tokenizer(model_name)
    if torch.cuda.is_available():
        print(f'GPU memory after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

    for experiment, task_tag, mod in TASKS:
        out_dir = ROOT / experiment / full_nick
        out_dir.mkdir(parents=True, exist_ok=True)

        # Skip if already done. Inference scripts write '..._{task_tag}_{short_nick}.jsonl'
        # (the file's nickname is the model basename, NOT the full thinking nickname —
        # the per-nickname directory is what disambiguates configs).
        existing = [p for p in out_dir.glob(f'*_{task_tag}_{short_nick}.jsonl')
                    if p.stat().st_size > MIN_VALID_OUTPUT_BYTES]
        if existing:
            print(f'  SKIP {full_nick} / {task_tag} (found {existing[0].name})')
            continue

        args = make_args(model_name, out_dir, cfg)
        print(f'\n--- {experiment} / {task_tag} ---')
        try:
            mod.run(model, tokenizer, args)
        except Exception as e:
            print(f'  FAILED {full_nick} / {task_tag}: {e}')
            traceback.print_exc()

    # Combined CSVs for this (model, config). Pass full_nick as --model so the
    # output CSV is named e.g. demographic_bias_processed_Qwen3-8B_thinkon.csv.
    for experiment, build_csv_path in BUILD_CSV_SCRIPTS.items():
        processed_dir = ROOT / experiment / 'processed'
        processed_dir.mkdir(parents=True, exist_ok=True)
        print(f'\n--- build_csv: {experiment} / {full_nick} ---')
        result = subprocess.run([
            'python', build_csv_path,
            '--model', full_nick,
            '--data-path', str(ROOT / experiment / full_nick),
            '--output-path', str(processed_dir),
        ])
        if result.returncode != 0:
            print(f'  WARNING: build_csv {experiment}/{full_nick} returned {result.returncode}')

    # Free GPU memory before next (model, config) tuple
    del model, tokenizer
    clear_gpu_memory()
    if torch.cuda.is_available():
        print(f'GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

print(f"\n{'='*70}\nALL DONE\n{'='*70}")